# 🎬 SmartStream AI — Entertainment Analytics Notebook
**AI-Powered Entertainment Recommendation System**  
Full exploratory data analysis, visualizations, and recommendation insights on 1000-row dataset.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# SmartStream brand colours
SS_BLUE   = '#378ADD'
SS_PINK   = '#D4537E'
SS_PURPLE = '#7F77DD'
SS_TEAL   = '#1D9E75'
SS_AMBER  = '#BA7517'
SS_CORAL  = '#D85A30'
PALETTE   = [SS_BLUE, SS_PINK, SS_PURPLE, SS_TEAL, SS_AMBER, SS_CORAL,
             '#639922', '#A32D2D', '#185FA5', '#993556', '#0F6E56', '#854F0B']

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#FAFAFA',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

print('✅ Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('smartstream_dataset.csv')
df['watch_date'] = pd.to_datetime(df['watch_date'])

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.describe()

In [ ]:
# Data quality check
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'\nData types:')
print(df.dtypes)

## 3. Overview Statistics

In [ ]:
stats = {
    'Total watch events':        len(df),
    'Unique users':              df['user_id'].nunique(),
    'Unique titles':             df['title'].nunique(),
    'Avg AI match score':        f"{df['ai_match_score'].mean():.1f}%",
    'Avg user rating':           f"{df['user_rating'].mean():.2f} / 5",
    'Completion rate':           f"{df['completed'].mean()*100:.1f}%",
    'Binge rate':                f"{df['is_binge'].mean()*100:.1f}%",
    'Recommendation-driven views':f"{df['was_recommended'].mean()*100:.1f}%",
    'Total watch hours':         f"{df['duration_minutes'].sum()/60:,.0f} h",
    'Avg session duration':      f"{df['duration_minutes'].mean():.0f} min",
}
pd.DataFrame.from_dict(stats, orient='index', columns=['Value'])

## 4. Genre Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SmartStream — Genre Analysis', fontsize=15, fontweight='bold', color='#222')

# Bar chart: genre counts
genre_counts = df['genre'].value_counts()
axes[0].barh(genre_counts.index[::-1], genre_counts.values[::-1], color=PALETTE[:len(genre_counts)])
axes[0].set_title('View Count by Genre')
axes[0].set_xlabel('Number of Views')
for i, v in enumerate(genre_counts.values[::-1]):
    axes[0].text(v+1, i, str(v), va='center', fontsize=9)

# Pie chart: genre share
wedges, texts, autotexts = axes[1].pie(
    genre_counts.values, labels=genre_counts.index,
    autopct='%1.1f%%', colors=PALETTE[:len(genre_counts)],
    startangle=140, pctdistance=0.82,
    wedgeprops=dict(linewidth=0.5, edgecolor='white')
)
for at in autotexts:
    at.set_fontsize(8)
axes[1].set_title('Genre Share (%)')

plt.tight_layout()
plt.savefig('chart_genre.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Platform Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SmartStream — Platform Insights', fontsize=15, fontweight='bold', color='#222')

plat_colors = {'Netflix': SS_PINK, 'Amazon Prime': SS_BLUE, 'Disney+': SS_AMBER, 'Apple TV+': '#888', 'HBO Max': SS_PURPLE}
plat_c = [plat_colors[p] for p in df['platform'].value_counts().index]

# 1. Count
plat_counts = df['platform'].value_counts()
axes[0].bar(plat_counts.index, plat_counts.values, color=plat_c)
axes[0].set_title('Views per Platform')
axes[0].set_ylabel('Views')
axes[0].tick_params(axis='x', rotation=20)

# 2. Avg rating per platform
plat_rating = df.groupby('platform')['user_rating'].mean().sort_values(ascending=False)
axes[1].barh(plat_rating.index, plat_rating.values, color=[plat_colors[p] for p in plat_rating.index])
axes[1].set_title('Avg User Rating by Platform')
axes[1].set_xlim(0, 5.5)
for i, v in enumerate(plat_rating.values):
    axes[1].text(v+0.05, i, f'{v:.2f}', va='center', fontsize=9)

# 3. Completion rate per platform
plat_comp = df.groupby('platform')['completed'].mean().sort_values(ascending=False) * 100
axes[2].barh(plat_comp.index, plat_comp.values, color=[plat_colors[p] for p in plat_comp.index])
axes[2].set_title('Completion Rate by Platform (%)')
axes[2].set_xlim(0, 115)
for i, v in enumerate(plat_comp.values):
    axes[2].text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_platform.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Viewing Behaviour — Time Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SmartStream — Viewing Behaviour', fontsize=15, fontweight='bold', color='#222')

# 1. Views by hour
hour_counts = df['watch_hour'].value_counts().sort_index()
axes[0,0].fill_between(hour_counts.index, hour_counts.values, alpha=0.3, color=SS_BLUE)
axes[0,0].plot(hour_counts.index, hour_counts.values, color=SS_BLUE, linewidth=2, marker='o', markersize=3)
axes[0,0].set_title('Viewing Activity by Hour of Day')
axes[0,0].set_xlabel('Hour (0–23)')
axes[0,0].set_ylabel('Views')
axes[0,0].set_xticks(range(0,24,2))

# 2. Views by day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['watch_day'].value_counts().reindex(day_order)
colors_day = [SS_CORAL if d in ['Saturday','Sunday'] else SS_PURPLE for d in day_order]
axes[0,1].bar(day_counts.index, day_counts.values, color=colors_day)
axes[0,1].set_title('Views by Day of Week')
axes[0,1].tick_params(axis='x', rotation=30)
axes[0,1].set_ylabel('Views')

# 3. Views by month
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_counts = df['watch_month'].value_counts().reindex(month_order)
axes[1,0].bar(range(12), month_counts.values, color=SS_TEAL, alpha=0.85)
axes[1,0].set_xticks(range(12))
axes[1,0].set_xticklabels([m[:3] for m in month_order], rotation=30)
axes[1,0].set_title('Monthly Viewing Volume')
axes[1,0].set_ylabel('Views')

# 4. Duration histogram
axes[1,1].hist(df['duration_minutes'], bins=30, color=SS_AMBER, edgecolor='white', linewidth=0.5)
axes[1,1].axvline(df['duration_minutes'].mean(), color=SS_CORAL, linestyle='--', linewidth=2,
                  label=f"Mean: {df['duration_minutes'].mean():.0f} min")
axes[1,1].set_title('Session Duration Distribution')
axes[1,1].set_xlabel('Duration (minutes)')
axes[1,1].set_ylabel('Frequency')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('chart_behaviour.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. AI Match Score & User Ratings Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SmartStream — AI Recommendation Intelligence', fontsize=15, fontweight='bold', color='#222')

# 1. AI match score distribution
axes[0,0].hist(df['ai_match_score'], bins=25, color=SS_BLUE, edgecolor='white', linewidth=0.5)
axes[0,0].axvline(df['ai_match_score'].mean(), color=SS_PINK, linestyle='--', linewidth=2,
                  label=f"Mean: {df['ai_match_score'].mean():.1f}%")
axes[0,0].set_title('AI Match Score Distribution')
axes[0,0].set_xlabel('AI Match Score (%)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].legend()

# 2. AI score vs user rating (scatter)
scatter_colors = [PALETTE[int(r)-1] for r in df['user_rating']]
axes[0,1].scatter(df['ai_match_score'], df['user_rating'],
                  c=scatter_colors, alpha=0.35, s=18, edgecolors='none')
z = np.polyfit(df['ai_match_score'], df['user_rating'], 1)
p = np.poly1d(z)
xline = np.linspace(df['ai_match_score'].min(), df['ai_match_score'].max(), 100)
axes[0,1].plot(xline, p(xline), color=SS_CORAL, linewidth=2, linestyle='--', label='Trend')
axes[0,1].set_title('AI Match Score vs User Rating')
axes[0,1].set_xlabel('AI Match Score (%)')
axes[0,1].set_ylabel('User Rating (1–5)')
axes[0,1].legend()

# 3. Rating distribution bar
rating_counts = df['user_rating'].value_counts().sort_index()
bar_colors = ['#E24B4A','#EF9F27','#BA7517','#1D9E75','#0F6E56']
bars = axes[1,0].bar(rating_counts.index, rating_counts.values, color=bar_colors, edgecolor='white')
axes[1,0].set_title('User Rating Distribution')
axes[1,0].set_xlabel('Rating')
axes[1,0].set_ylabel('Count')
for bar, val in zip(bars, rating_counts.values):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                   str(val), ha='center', va='bottom', fontsize=9)

# 4. Avg AI score by genre
genre_ai = df.groupby('genre')['ai_match_score'].mean().sort_values(ascending=False)
axes[1,1].barh(genre_ai.index, genre_ai.values, color=PALETTE[:len(genre_ai)])
axes[1,1].set_title('Avg AI Match Score by Genre')
axes[1,1].set_xlabel('Avg Match Score (%)')
axes[1,1].set_xlim(70, 100)
for i, v in enumerate(genre_ai.values):
    axes[1,1].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_ai_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Device & Demographics Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SmartStream — Device & Demographics', fontsize=15, fontweight='bold', color='#222')

# 1. Device breakdown
dev_counts = df['device'].value_counts()
axes[0].pie(dev_counts.values, labels=dev_counts.index, autopct='%1.1f%%',
            colors=PALETTE[:len(dev_counts)], startangle=90,
            wedgeprops=dict(linewidth=0.6, edgecolor='white'))
axes[0].set_title('Device Usage')

# 2. Age group
age_order = ['13-17','18-24','25-34','35-44','45-54','55+']
age_counts = df['age_group'].value_counts().reindex(age_order)
axes[1].bar(age_counts.index, age_counts.values, color=SS_PURPLE, alpha=0.85, edgecolor='white')
axes[1].set_title('Viewers by Age Group')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=20)

# 3. Subscription tier
tier_order = ['Free','Basic','Standard','Premium']
tier_counts = df['subscription_tier'].value_counts().reindex(tier_order)
tier_colors = ['#B4B2A9', SS_TEAL, SS_BLUE, SS_AMBER]
axes[2].bar(tier_counts.index, tier_counts.values, color=tier_colors, edgecolor='white')
axes[2].set_title('Subscription Tier Breakdown')
axes[2].set_ylabel('Count')
for i, (idx, v) in enumerate(zip(tier_counts.index, tier_counts.values)):
    axes[2].text(i, v+3, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Country & Language Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SmartStream — Geographic & Language Insights', fontsize=15, fontweight='bold', color='#222')

country_counts = df['country'].value_counts()
axes[0].barh(country_counts.index[::-1], country_counts.values[::-1], color=SS_TEAL)
axes[0].set_title('Views by Country')
axes[0].set_xlabel('Number of Views')
for i, v in enumerate(country_counts.values[::-1]):
    axes[0].text(v+1, i, str(v), va='center', fontsize=9)

lang_counts = df['language'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    lang_counts.values, labels=lang_counts.index, autopct='%1.1f%%',
    colors=PALETTE[:len(lang_counts)], startangle=120,
    wedgeprops=dict(linewidth=0.5, edgecolor='white')
)
for at in autotexts: at.set_fontsize(8)
axes[1].set_title('Content Language Distribution')

plt.tight_layout()
plt.savefig('chart_geo.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Correlation Heatmap

In [ ]:
numeric_cols = ['duration_minutes','user_rating','ai_match_score','watch_hour',
                'release_year','imdb_score','completed','is_binge',
                'was_recommended','trailer_watched','added_to_watchlist','shared_content']

corr_df = df[numeric_cols].copy()
bool_cols = ['completed','is_binge','was_recommended','trailer_watched','added_to_watchlist','shared_content']
corr_df[bool_cols] = corr_df[bool_cols].astype(int)

corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=0.4, vmin=-0.4, center=0,
            square=True, linewidths=0.5, annot=True, fmt='.2f', ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix — SmartStream Dataset', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=40)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('chart_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Content Type Deep Dive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SmartStream — Content Type Analysis', fontsize=15, fontweight='bold', color='#222')

ct_order = df['content_type'].value_counts().index
ct_colors = [PALETTE[i] for i in range(len(ct_order))]

# Grouped bar: avg rating and avg AI score by type
x = np.arange(len(ct_order))
w = 0.35
r1 = df.groupby('content_type')['user_rating'].mean().reindex(ct_order)
r2 = df.groupby('content_type')['ai_match_score'].mean().reindex(ct_order) / 20  # scale to 0-5

axes[0].bar(x - w/2, r1.values, w, label='Avg User Rating', color=SS_BLUE, alpha=0.85)
axes[0].bar(x + w/2, r2.values, w, label='AI Score /20', color=SS_PINK, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(ct_order, rotation=15)
axes[0].set_title('Rating & AI Score by Content Type')
axes[0].legend()

# Completion rate by content type
comp_rate = df.groupby('content_type')['completed'].mean().reindex(ct_order) * 100
bars = axes[1].bar(comp_rate.index, comp_rate.values, color=ct_colors, edgecolor='white')
axes[1].set_title('Completion Rate by Content Type (%)')
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 120)
axes[1].tick_params(axis='x', rotation=15)
for bar, v in zip(bars, comp_rate.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%',
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('chart_content_type.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Engagement Funnel

In [ ]:
funnel_stages = {
    'Content Discovered':     len(df),
    'Trailer Watched':        df['trailer_watched'].sum(),
    'Added to Watchlist':     df['added_to_watchlist'].sum(),
    'Watch Started':          len(df),
    'Watch Completed':        df['completed'].sum(),
    'Rated by User':          len(df),
    'Shared Content':         df['shared_content'].sum(),
}
# reset started = all
funnel_stages['Watch Started'] = funnel_stages['Content Discovered']
funnel_stages['Rated by User'] = df['completed'].sum()

fig, ax = plt.subplots(figsize=(10, 6))
stages = list(funnel_stages.keys())
values = list(funnel_stages.values())
max_v = max(values)
bar_colors = [SS_BLUE, SS_PURPLE, SS_TEAL, SS_AMBER, SS_CORAL, SS_PINK, '#639922']

for i, (s, v, c) in enumerate(zip(stages, values, bar_colors)):
    left = (max_v - v) / 2
    ax.barh(i, v, left=left, color=c, alpha=0.85, height=0.6, edgecolor='white')
    ax.text(max_v/2, i, f'{s}  ({v:,} — {v/max_v*100:.0f}%)', 
            ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xlim(0, max_v)
ax.axis('off')
ax.set_title('SmartStream — User Engagement Funnel', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Top Titles Leaderboard

In [ ]:
top_titles = (df.groupby('title').agg(
    views=('title','count'),
    avg_rating=('user_rating','mean'),
    avg_ai_score=('ai_match_score','mean'),
    completion_rate=('completed','mean'),
    total_hours=('duration_minutes', lambda x: x.sum()/60)
).sort_values('views', ascending=False).head(15).round(2))

top_titles['completion_rate'] = (top_titles['completion_rate']*100).round(1).astype(str) + '%'
top_titles['total_hours'] = top_titles['total_hours'].round(1)
print('🏆 Top 15 Titles by Views')
top_titles

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
top15 = df.groupby('title')['title'].count().sort_values(ascending=False).head(15)
colors = [PALETTE[i % len(PALETTE)] for i in range(len(top15))]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1])
ax.set_title('Top 15 Most Watched Titles', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Views')
for i, v in enumerate(top15.values[::-1]):
    ax.text(v+0.3, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('chart_top_titles.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Subscription Tier Behaviour

In [ ]:
tier_analysis = df.groupby('subscription_tier').agg(
    avg_rating=('user_rating','mean'),
    avg_ai_score=('ai_match_score','mean'),
    binge_rate=('is_binge','mean'),
    completion_rate=('completed','mean'),
    avg_duration=('duration_minutes','mean'),
    share_rate=('shared_content','mean')
).reindex(['Free','Basic','Standard','Premium']).round(3)

tier_analysis[['binge_rate','completion_rate','share_rate']] = \
    (tier_analysis[['binge_rate','completion_rate','share_rate']] * 100).round(1)

print('Subscription Tier Behaviour Matrix')
tier_analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Behaviour by Subscription Tier', fontsize=13, fontweight='bold')
tiers = ['Free','Basic','Standard','Premium']
tier_colors2 = ['#B4B2A9', SS_TEAL, SS_BLUE, SS_AMBER]

binge = tier_analysis['binge_rate'].values
comp  = tier_analysis['completion_rate'].values
x = np.arange(len(tiers))
w = 0.35
axes[0].bar(x-w/2, binge, w, label='Binge Rate %', color=SS_PURPLE, alpha=0.85)
axes[0].bar(x+w/2, comp, w, label='Completion Rate %', color=SS_TEAL, alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(tiers)
axes[0].set_title('Engagement Rates')
axes[0].legend(fontsize=9)

axes[1].bar(tiers, tier_analysis['avg_duration'].values, color=tier_colors2, edgecolor='white')
axes[1].set_title('Avg Session Duration (min)')
axes[1].set_ylabel('Minutes')
for i, v in enumerate(tier_analysis['avg_duration'].values):
    axes[1].text(i, v+0.5, f'{v:.0f}m', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_tiers.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. IMDB Score vs User Rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('SmartStream — IMDB vs User Intelligence', fontsize=13, fontweight='bold')

genre_colors_map = {g: PALETTE[i] for i, g in enumerate(df['genre'].unique())}
sc_colors = [genre_colors_map[g] for g in df['genre']]
axes[0].scatter(df['imdb_score'], df['user_rating'], c=sc_colors, alpha=0.3, s=16, edgecolors='none')
z = np.polyfit(df['imdb_score'], df['user_rating'], 1)
xline = np.linspace(df['imdb_score'].min(), df['imdb_score'].max(), 100)
axes[0].plot(xline, np.poly1d(z)(xline), color=SS_CORAL, linewidth=2, linestyle='--')
axes[0].set_title('IMDB Score vs User Rating')
axes[0].set_xlabel('IMDB Score')
axes[0].set_ylabel('User Rating')

imdb_bins = pd.cut(df['imdb_score'], bins=[1,4,6,7,8,10], labels=['1-4','4-6','6-7','7-8','8-10'])
imdb_rating = df.groupby(imdb_bins)['user_rating'].mean()
axes[1].bar(imdb_rating.index.astype(str), imdb_rating.values, color=SS_BLUE, edgecolor='white')
axes[1].set_title('Avg User Rating by IMDB Bracket')
axes[1].set_xlabel('IMDB Score Range')
axes[1].set_ylabel('Avg User Rating')
axes[1].set_ylim(0, 6)
for i, v in enumerate(imdb_rating.values):
    axes[1].text(i, v+0.05, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_imdb.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. Summary Report

In [ ]:
print('='*60)
print('       SMARTSTREAM AI — ANALYTICS SUMMARY REPORT')
print('='*60)
print(f'\n📊 Dataset: {len(df)} events | {df["user_id"].nunique()} users | {df["title"].nunique()} titles')
print(f'\n🎯 AI Performance:')
print(f'   Avg match score:     {df["ai_match_score"].mean():.1f}%')
print(f'   Score > 90%:         {(df["ai_match_score"]>90).mean()*100:.1f}% of recommendations')
print(f'   Rec-driven views:    {df["was_recommended"].mean()*100:.1f}%')
print(f'\n📺 Viewing Behaviour:')
print(f'   Completion rate:     {df["completed"].mean()*100:.1f}%')
print(f'   Binge rate:          {df["is_binge"].mean()*100:.1f}%')
print(f'   Avg session:         {df["duration_minutes"].mean():.0f} min')
print(f'   Peak hour:           {df["watch_hour"].mode()[0]:02d}:00')
print(f'   Top day:             {df["watch_day"].mode()[0]}')
print(f'\n⭐ Content Quality:')
print(f'   Avg user rating:     {df["user_rating"].mean():.2f}/5')
print(f'   Avg IMDB score:      {df["imdb_score"].mean():.1f}/10')
print(f'   Top genre:           {df["genre"].mode()[0]}')
print(f'   Top platform:        {df["platform"].mode()[0]}')
print(f'   Top content type:    {df["content_type"].mode()[0]}')
print(f'\n🌍 Audience:')
print(f'   Top country:         {df["country"].mode()[0]}')
print(f'   Top age group:       {df["age_group"].mode()[0]}')
print(f'   Top tier:            {df["subscription_tier"].mode()[0]}')
print(f'\n📈 Engagement:')
print(f'   Trailer watch rate:  {df["trailer_watched"].mean()*100:.1f}%')
print(f'   Watchlist adds:      {df["added_to_watchlist"].mean()*100:.1f}%')
print(f'   Share rate:          {df["shared_content"].mean()*100:.1f}%')
print('='*60)
print('✅ Analysis complete — SmartStream AI')